Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  str    
 2   color    53943 non-null  str    
 3   clarity  53943 non-null  str    
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I     VS2   

Tiền xử lý dữ liệu

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia dữ liệu

In [3]:
from sklearn.model_selection import train_test_split

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu train test
X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size=0.2,
      random_state=42
    )

# In ra kết quả chia dữ liệu
print(f"Có {X_train.shape[0]} mẫu huấn luyện và {X_test.shape[0]} mẫu kiểm tra")

Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra


Chuẩn hoá dữ liệu

In [4]:
# Import công cụ chuẩn hóa MinMaxScaler để đưa các đặc trưng về khoảng [0, 1]
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo đối tượng MinMaxScaler
scaler = MinMaxScaler()
# Học tham số Min-Max trên tập Train và chuyển đổi X_train sang dạng chuẩn hóa [0, 1]
X_train_scaled = scaler.fit_transform(X_train)
# Chuyển đổi X_test dựa trên các tham số Min-Max đã học từ X_train (tránh rò rỉ dữ liệu)
X_test_scaled = scaler.transform(X_test)

Training và kết quả đánh giá

In [ ]:
# Import mô hình hồi quy K-Nearest Neighbors (KNN) từ thư viện scikit-learn
from sklearn.neighbors import KNeighborsRegressor
# Import các hàm tính toán độ đo đánh giá hiệu năng mô hình (MAE, MSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import thư viện numpy để thực hiện các phép toán số học (như tính căn bậc hai)
import numpy as np
# Import thư viện time để đo thời gian thực thi của mô hình
import time

# 1. THAM SỐ MỞ RỘNG (KHOẢNG CÁCH MANHATTAN)
# Tạo danh sách các giá trị số lượng láng giềng K cần thử nghiệm (từ 1 đến 100)
k_values = range(1, 101)
# Tạo danh sách các phương pháp tính trọng số láng giềng ('uniform': đều nhau, 'distance': theo khoảng cách)
weights_list = ['uniform', 'distance']

# Khởi tạo biến theo dõi sai số RMSE thấp nhất trên tập Test (gán ban đầu bằng dương vô cực)
best_rmse = float('inf')
# Khởi tạo biến lưu trữ đối tượng mô hình tốt nhất tìm được
best_model = None
# Khởi tạo biến lưu trữ từ điển bộ tham số tối ưu nhất
best_params = None
# Khởi tạo biến lưu trữ mảng kết quả dự đoán trên tập Test của mô hình tốt nhất
best_y_pred_test = None
# Khởi tạo biến lưu trữ thời gian thực thi (fit + predict Test) của mô hình tốt nhất
best_time = 0.0

# In thông báo bắt đầu quá trình duyệt tìm tham số tối ưu
print("--- Đang tìm kiếm tham số tối ưu cho KNN (K từ 1 đến 100, khoảng cách Manhattan) ---")

# 2. DUYỆT QUA CÁC TỔ HỢP THAM SỐ
# Vòng lặp ngoài duyệt qua từng kiểu trọng số trong danh sách weights_list
for w in weights_list:
    # Vòng lặp trong duyệt qua từng giá trị số lượng láng giềng K từ 1 đến 100
    for k in k_values:
        # Khởi tạo mô hình KNN với K và trọng số hiện tại, cố định khoảng cách Manhattan (metric='minkowski', p=1)
        knn = KNeighborsRegressor(n_neighbors=k, weights=w, metric='minkowski', p=1)

        # Bắt đầu bấm giờ bằng hàm perf_counter() trước khi huấn luyện và dự đoán
        start_time = time.perf_counter()
        # Huấn luyện mô hình KNN trên tập dữ liệu Train đã chuẩn hóa (X_train_scaled) và nhãn (y_train)
        knn.fit(X_train_scaled, y_train)
        # Dự đoán giá trị nhãn (giá kim cương) trên tập dữ liệu Test đã chuẩn hóa (X_test_scaled)
        y_pred = knn.predict(X_test_scaled)
        # Kết thúc bấm giờ và tính tổng thời gian thực thi (fit + predict Test) bằng đơn vị giây
        elapsed_time = time.perf_counter() - start_time

        # Tính sai số RMSE trên tập Test bằng cách lấy căn bậc hai của sai số bình phương trung bình (MSE)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        # Kiểm tra nếu RMSE của tổ hợp tham số hiện tại nhỏ hơn mức RMSE tốt nhất từng ghi nhận
        if rmse < best_rmse:
            # Cập nhật giá trị RMSE tốt nhất mới
            best_rmse = rmse
            # Lưu lại đối tượng mô hình tốt nhất hiện tại
            best_model = knn
            # Lưu lại bộ tham số tương ứng của mô hình tốt nhất
            best_params = {'n_neighbors': k, 'weights': w, 'metric': 'minkowski', 'p': 1}
            # Lưu lại mảng dự đoán trên tập Test của mô hình tốt nhất
            best_y_pred_test = y_pred
            # Lưu lại thời gian thực thi của mô hình tốt nhất
            best_time = elapsed_time

# 3. IN THÔNG TIN VÀ KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT
# In ra bộ tham số đạt được sai số RMSE thấp nhất trên tập Test
print("\nBỘ THAM SỐ TỐI ƯU NHẤT:", best_params)

# Dùng mô hình tốt nhất để dự đoán giá trị nhãn trên tập dữ liệu Train đã chuẩn hóa
y_pred_train = best_model.predict(X_train_scaled)

# In tiêu đề thông báo phần đánh giá kết quả trên tập huấn luyện Train
print("\n=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TRAIN ===")
# Tính sai số tuyệt đối trung bình (MAE) trên tập Train và in ra kết quả làm tròn 4 chữ số thập phân
print("MAE:", round(mean_absolute_error(y_train, y_pred_train), 4))
# Tính sai số bình phương trung bình (MSE) trên tập Train và in ra kết quả làm tròn 4 chữ số thập phân
print("MSE:", round(mean_squared_error(y_train, y_pred_train), 4))
# Tính căn bậc hai của MSE (RMSE) trên tập Train và in ra kết quả làm tròn 4 chữ số thập phân
print("RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 4))
# Tính hệ số xác định R-squared (R^2) trên tập Train và in ra kết quả làm tròn 4 chữ số thập phân
print("R^2:", round(r2_score(y_train, y_pred_train), 4))

# In tiêu đề thông báo phần đánh giá kết quả trên tập kiểm tra Test
print("\n=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TEST ===")
# Tính sai số tuyệt đối trung bình (MAE) trên tập Test và in ra kết quả làm tròn 4 chữ số thập phân
print("MAE:", round(mean_absolute_error(y_test, best_y_pred_test), 4))
# Tính sai số bình phương trung bình (MSE) trên tập Test và in ra kết quả làm tròn 4 chữ số thập phân
print("MSE:", round(mean_squared_error(y_test, best_y_pred_test), 4))
# In ra giá trị RMSE thấp nhất trên tập Test đã lưu được (làm tròn 4 chữ số thập phân)
print("RMSE:", round(best_rmse, 4))
# Tính hệ số xác định R-squared (R^2) trên tập Test và in ra kết quả làm tròn 4 chữ số thập phân
print("R^2:", round(r2_score(y_test, best_y_pred_test), 4))
# In ra tổng thời gian thực thi (huấn luyện và dự đoán tập Test) của mô hình tối ưu làm tròn 4 chữ số thập phân
print(f"Thời gian thực thi (fit + predict Test): {best_time:.4f} giây")

--- Đang tìm kiếm tham số tối ưu cho KNN (K từ 1 đến 100, khoảng cách Manhattan) ---

BỘ THAM SỐ TỐI ƯU NHẤT: {'n_neighbors': 8, 'weights': 'distance', 'metric': 'minkowski', 'p': 1}

=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TRAIN ===
MAE: 0.4436
MSE: 78.5161
RMSE: 8.8609
R^2: 1.0

=== KẾT QUẢ MÔ HÌNH TỐI ƯU TRÊN TẬP TEST ===
MAE: 318.5839
MSE: 389816.7677
RMSE: 624.3531
R^2: 0.9753
Thời gian thực thi (fit + predict Test): 0.6847 giây
